# Imports

In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime

import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns

from math import sqrt 
from sklearn.metrics import mean_squared_error

from statsmodels.tsa.api import ExponentialSmoothing, SimpleExpSmoothing
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose

import sys
import os
sys.path.insert(0, os.path.abspath(".."))

from funcs import test_stationarity, diff

In [ ]:
pd.set_option('display.expand_frame_repr', False)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

In [ ]:
mpl.rcParams['axes.labelsize'] = 14
mpl.rcParams['xtick.labelsize'] = 12
mpl.rcParams['ytick.labelsize'] = 12
mpl.rcParams['text.color'] = 'k'
mpl.rcParams['figure.figsize'] = (9,5)
plt.style.use('ggplot')

# Load Data

In [ ]:
df = pd.read_csv("../data/raw/dataset_part2.csv",
                          index_col=1,
                          parse_dates=True,
                          date_format="%d-%m-%Y %H:%M",
                          header=0)

In [ ]:
df.index

In [ ]:
df.info()

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df.drop(labels='id', axis=1,inplace=True)

In [ ]:
df.head()

In [ ]:
df['active_users'].plot()
plt.show()

In [ ]:
# Plot one day

df['active_users'][:25].plot()
plt.show()

# Pre-Processing

In [ ]:
df_train = df[:16057]
df_valid = df[16057:]

In [ ]:
df_train['log'] = np.log(df_train['active_users'])
df_valid['log'] = np.log(df_valid['active_users'])

In [ ]:
plt.plot(df_train['log'], label = "Train Data")
plt.plot(df_valid['log'], label = "Validation Data")
plt.legend()
plt.show()

In [ ]:
test_stationarity(df['active_users'])

In [ ]:
arr_train = df_train['log'].to_numpy()
arr_valid = df_valid['log'].to_numpy()

# Forecasting

In [ ]:
len_forescast = len(df_valid)
len_forescast

## Simple Exponential Smoothing

In [ ]:
model_ses = SimpleExpSmoothing(endog=arr_train)

In [ ]:
model_ses_res = model_ses.fit(optimized=True)

In [ ]:
model_ses_res.params

In [ ]:
df_valid['ses_pred'] = model_ses_res.forecast(len_forescast)

In [ ]:
print(f"RMSE using Simple Exponential Smoothing Forecast:\n"
      f"{np.sqrt(mean_squared_error(df_valid['log'], df_valid['ses_pred'])):.4f}")

In [ ]:
plt.plot(df_train['log'])
plt.plot(df_valid['log'])
plt.plot(df_valid['ses_pred'], c='g')
plt.show()

## Double Exponential smoothing (Holt linear)

- When using the trend hyperparameter, we define the Double Exponential Smoothing method 
- When using the seasonal hyperparameter, we define the Triple Exponential Smoothing method.

In [ ]:
model_des = ExponentialSmoothing(endog=arr_train, trend='add')

In [ ]:
model_des_res = model_des.fit(optimized=True)

In [ ]:
model_des_res.params

In [ ]:
df_valid['des_pred'] = model_des_res.forecast(len_forescast)

In [ ]:
plt.plot(df_train['log'])
plt.plot(df_valid['log'])
plt.plot(df_valid['ses_pred'], c='g')
plt.plot(df_valid['des_pred'], c='b')
plt.show()

In [ ]:
print(f"RMSE using Double Exponential Smoothing Forecast:\n"
      f"{np.sqrt(mean_squared_error(df_valid['log'], df_valid['des_pred'])):.4f}")

## Triple

In [ ]:
# seasonal_periods = day

model_tes = ExponentialSmoothing(endog=arr_train, trend='add', seasonal='add', seasonal_periods=24)

In [ ]:
model_tes_res = model_tes.fit(optimized=True)

In [ ]:
model_tes_res.params

In [ ]:
df_valid['tes_pred'] = model_tes_res.forecast(len_forescast)

In [ ]:
plt.plot(df_valid['log'])
plt.plot(df_valid['tes_pred'], c='b')
plt.show()

In [ ]:
print(f"RMSE using Triple Exponential Smoothing Forecast:\n"
      f"{np.sqrt(mean_squared_error(df_valid['log'], df_valid['tes_pred'])):.4f}")